Imports

In [ ]:
import notebook
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
from platform import python_version
import xgboost as xgb
import graphviz
from datetime import time
from datetime import datetime
from datetime import timedelta, date
import sklearn
from sklearn.model_selection import LearningCurveDisplay
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.inspection import permutation_importance
import itertools


print(f"python: v {python_version()}")
print(f"Jupyter Notebook: v {notebook.__version__}")
print(f"numpy: v {np.__version__}")
print(f"pandas: v {pd.__version__}")
print(f"seaborn: v {sns.__version__}")
print(f"graphviz: v {graphviz.__version__}")
print(f"matplotlib: v {matplotlib.__version__}")
print(f"sklearn: v {sklearn.__version__}")
print(f"XGBoost: v {xgb.__version__}")

python: v 3.10.12
Jupyter Notebook: v 7.4.7
numpy: v 1.25.2
pandas: v 2.3.3
seaborn: v 0.13.2
graphviz: v 0.21
matplotlib: v 3.7.2
sklearn: v 1.7.2
XGBoost: v 3.1.1


Import Data

In [ ]:
# house_num = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 21]
house_num = [12, 13, 18]
df_hourlys = []

mapping_12 = {
    'Appliance1': 5,
    'Appliance2': 5,
    'Appliance3': 20,
    'Appliance4': 5,
    'Appliance5': 10,
    'Appliance6': 20
}
mapping_13 = {
    'Appliance1': 20,
    'Appliance2': 15,
    'Appliance3': 20,
    'Appliance4': 10,
    'Appliance5': 150,
    'Appliance6': 10,
    'Appliance7': 10,
    'Appliance8': 30,
    'Appliance9': 1
}
mapping_18 = {
    'Appliance1': 50,
    'Appliance2': 10,
    'Appliance3': 10,
    'Appliance4': 25,
    'Appliance5': 20,
    'Appliance6': 20,
    'Appliance7': 25,
    'Appliance8': 30,
    'Appliance9': 25
}
APPLIANCE_COLS = [
    'Appliance1','Appliance2', 'Appliance3',
    'Appliance4', 'Appliance5', 'Appliance6',
    'Appliance7', 'Appliance8', 'Appliance9'
]
thresh = 5

for selected in house_num:
    df = pd.read_csv(f"../data/Refit/CLEAN_House{selected}.csv", header=0)
    df.rename(columns={'Time': 'Date'}, inplace=True)
    df.rename(columns={'Aggregate': 'Volume'}, inplace=True)
    df = df.dropna(subset=['Date', 'Volume'])
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce').dt.tz_localize(None)
    df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce')
    df = df.set_index('Date')
    hourly = df['Volume'].resample('1h').mean()
    df_hourly = hourly.to_frame(name='Volume').reset_index()
    if selected == 12:
        APPLIANCEs = ['Appliance1','Appliance2', 'Appliance3', 'Appliance4', 'Appliance5', 'Appliance6']
    else:
        APPLIANCEs = APPLIANCE_COLS
    df_nilm = df[APPLIANCEs].resample('1h').mean().reset_index()
    for device in APPLIANCEs:
        if selected == 13:
            thresh = mapping_13[device]
        elif selected == 18:
            thresh = mapping_18[device]
        elif selected == 12:
            thresh = mapping_12[device]
        df_nilm[device] = pd.to_numeric(df_nilm[device], errors='coerce').fillna(0)
        df_nilm[device] = df_nilm[device].apply(lambda x: 1 if x > thresh else 0)
    df_hourly = pd.merge(df_hourly, df_nilm, on='Date', how='left')
    df_hourlys.append(df_hourly)
    print(f"Processed House_{selected}")




print("CSVs read successfully.")

Processed House_12
CSVs read successfully.


Import Temperature Information

In [3]:
df_temps = pd.read_csv('../data/Refit/hourly_Temp_data.csv')
df_temps.rename(columns={'date': 'Date'}, inplace=True)
df_temps.rename(columns={'temperature_2m': 'Temperature'}, inplace=True)
df_temps['Date'] = pd.to_datetime(df_temps['Date'], errors='coerce').dt.tz_localize(None)
df_temps = df_temps.set_index('Date')
df_temps = df_temps.asfreq("1h")
df_temps = df_temps.dropna(subset=['Temperature'])
df_temps['Temperature'] = pd.to_numeric(df_temps['Temperature'], errors='coerce')
temp_indexs = {}
for i in range(len(df_hourlys)):
    df_hourlys[i] = pd.merge(df_hourlys[i], df_temps, on='Date', how='left')
    temp_indexs[i] = df_hourlys[i].set_index('Date')
    # print first and last timestamps :
    print(f"First timestamp in df_hourlys[{i}]: {df_hourlys[i]['Date'].min()}")
    print(f"Last timestamp in df_hourlys[{i}]: {df_hourlys[i]['Date'].max()}")

print("Temperature CSV read successfully.")


First timestamp in df_hourlys[0]: 2014-03-07 10:00:00
Last timestamp in df_hourlys[0]: 2015-07-08 02:00:00
Temperature CSV read successfully.


Feature Engineering

In [4]:
def add_date_scalar(df):
    df = df.copy()
    df['Year'] = df.Date.dt.year
    df['Month'] = df.Date.dt.month
    df['Day'] = df.Date.dt.day
    df['Hour'] = df.Date.dt.hour
    df['Weekday'] = df.Date.dt.weekday  
    return df


def get_type_day(row):
    if row['Weekday'] in (5, 6): 
        return 1
    return 0

def add_type_day(df):
    df = df.copy()
    df['TypeDay'] = df.apply(get_type_day, axis=1)
    return df

# Add lag features while respecting no information after 10:00 AM previous day (offset = 2)
def prepare_data_with_lags(df, lag_start = 1, lag_end = 7, base_offset_days = 2, dropna = True):
    df = df.copy()
    df = add_type_day(df)

    df = df.set_index('Date')

    for k in range(lag_start, lag_end + 1):
        days_back = base_offset_days + (k - 1)
        df[f"lag-{k}"] = df['Volume'].shift(freq=f"{days_back}D")

    if dropna:
        lag_cols = [f"lag-{k}" for k in range(lag_start, lag_end + 1)]
        df = df.dropna(subset=lag_cols)

    df = df.reset_index()
    return df

# Builds a DataFrame with 24 hours
def get_weather_for_day(date_, temp_index, drop_appliances = None, num=0):
    full_range = pd.date_range(date_, periods=25, freq='h')
    df_empty = pd.DataFrame({'Date': full_range})
    if num == 12:
        APP_COLS = ['Appliance1','Appliance2', 'Appliance3', 'Appliance4', 'Appliance5', 'Appliance6']
    else:
        APP_COLS = APPLIANCE_COLS
    for device in APP_COLS:
        df_empty[device] = 0

    if drop_appliances is not None:
        df_empty = df_empty.drop(columns=drop_appliances, errors='ignore')
    df_empty['Temperature'] = 0

    df = df_empty.tail(25).reset_index(drop=True)
    return df

Replacing NaN values

In [ ]:
group_median = ()
def fill_with_group_median(row):
    if pd.isna(row['Volume']):
        return group_median.loc[(row['Month'], row['Weekday'], row['Hour'])]
    else:
        return row['Volume']


df_generals = []
df_analytics = []
for i in range(len(df_hourlys)):
    df_generals.append(add_date_scalar(df_hourlys[i]))
    
    group_median = (
        df_generals[i]
        .groupby(['Month', 'Weekday', 'Hour'])['Volume']
        .median()
    )
    print(f"For House_{house_num[i]} :")

    # Data visualization before filling missing values filled with group median

    # plt.figure(figsize=(12, 6))
    # plt.plot(df_generals[i]['Date'], df_generals[i]['Volume'], label='Aggregate Power Consumption')
    # plt.xlabel('Date')
    # plt.ylabel('Power Consumption (W)')
    # plt.title('Aggregate Power Consumption Over Time')
    # plt.legend()
    # plt.show()

    # Number of missing values in Volume before filling with group median
    # missing_count = df_generals[i]['Volume'].isna().sum()
    # print(f"Number of missing values in Volume for House_{house_num[i]}: {missing_count} out of {len(df_generals[i])} ")

    df_generals[i]['Volume'] = df_generals[i].apply(fill_with_group_median, axis=1)
    
    # Data visualization after filling missing values filled with group median
    
    # plt.figure(figsize=(12, 6))
    # plt.plot(df_generals[i]['Date'], df_generals[i]['Volume'], label='Aggregate Power Consumption')
    # plt.xlabel('Date')
    # plt.ylabel('Power Consumption (W)')
    # plt.title('Aggregate Power Consumption Over Time')
    # plt.legend()
    # plt.show()
    
    df_analytics.append(df_generals[i].copy())
    df_generals[i] = prepare_data_with_lags(df_generals[i])


print("Data preparation completed.")



For House_12 :
Number of missing values in Volume for House_12: 720 out of 11705 


Data preparation completed.


Data Visualization helper functions

In [ ]:
def plot_volume_over_time(df_hourly):
    plt.figure(figsize=(14, 5))
    plt.plot(df_hourly['Date'], df_hourly['Volume'])
    plt.title('Hourly Volume Over Time')
    plt.xlabel('Date')
    plt.ylabel('Volume')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.close()

def plot_volume_and_temperature(df_hourly):
    fig, ax1 = plt.subplots(figsize=(14, 5))

    ax1.set_xlabel('Date')
    ax1.set_ylabel('Volume', color='tab:blue')
    ax1.plot(df_hourly['Date'], df_hourly['Volume'], color='tab:blue', label='Volume')
    ax1.tick_params(axis='y', labelcolor='tab:blue')

    ax2 = ax1.twinx()  
    ax2.set_ylabel('Temperature', color='tab:red')  
    ax2.plot(df_hourly['Date'], df_hourly['Temperature'], color='tab:red', label='Temperature')
    ax2.tick_params(axis='y', labelcolor='tab:red')

    plt.title('Volume and Temperature Over Time')
    fig.tight_layout()
    plt.show()
    plt.close()

def plot_avg_profile_by_hour(df_hourly):
    df = df_hourly.copy()
    df['Hour'] = df['Date'].dt.hour
    hourly_profile = df.groupby('Hour')['Volume'].mean()

    plt.figure(figsize=(8, 4))
    hourly_profile.plot(marker='o')
    plt.title('Average Daily Load Profile (by Hour)')
    plt.xlabel('Hour of Day')
    plt.ylabel('Average Volume')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.close()

def plot_avg_volume_by_weekday(df_hourly):
    df = df_hourly.copy()
    df['Weekday'] = df['Date'].dt.weekday  # 0=Mon, 6=Sun
    weekday_profile = df.groupby('Weekday')['Volume'].mean()

    plt.figure(figsize=(8, 4))
    weekday_profile.plot(kind='bar')
    plt.title('Average Volume by Weekday')
    plt.xlabel('Weekday (0=Mon)')
    plt.ylabel('Average Volume')
    plt.grid(axis='y')
    plt.tight_layout()
    plt.show()
    plt.close()


def plot_avg_volume_by_typeday(df_general):
    df = df_general.copy()
    if 'TypeDay' not in df.columns:
        df = add_type_day(df)

    typeday_profile = df.groupby('TypeDay')['Volume'].mean()

    plt.figure(figsize=(6, 4))
    typeday_profile.plot(kind='bar')
    plt.title('Average Volume by TypeDay')
    plt.xlabel('TypeDay')
    plt.ylabel('Average Volume')
    plt.grid(axis='y')
    plt.tight_layout()
    plt.show()
    plt.close()

def plot_volume_vs_temperature(df_hourly):
    df = df_hourly.copy()
    df = df.dropna(subset=['Temperature', 'Volume'])

    plt.figure(figsize=(6, 4))
    plt.scatter(df['Temperature'], df['Volume'], alpha=0.3)
    plt.xlim(0, 25)
    plt.ylim(-500, 6000)
    plt.title('Volume vs Temperature')
    plt.xlabel('Temperature')
    plt.ylabel('Volume')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.close()


def plot_acf_pacf_of_series(df_hourly, max_lags_hours=24*7):
    series = df_hourly['Volume'].dropna()

    plt.figure(figsize=(10, 4))
    plot_acf(series, lags=max_lags_hours)
    plt.title('ACF of Volume')
    plt.tight_layout()
    plt.show()
    plt.close()

    plt.figure(figsize=(10, 4))
    plot_pacf(series, lags=max_lags_hours)
    plt.title('PACF of Volume')
    plt.tight_layout()
    plt.show()
    plt.close()

Data visualization

In [7]:
# for i in range(len(df_analytics)):
#     num = house_num[i]
#     print(f"Generating plots for House_{num}:\n")
#     plot_volume_over_time(df_analytics[i])
#     plot_avg_profile_by_hour(df_analytics[i])
#     plot_avg_volume_by_weekday(df_analytics[i])
#     plot_avg_volume_by_typeday(df_analytics[i])
#     plot_volume_vs_temperature(df_analytics[i])
#     plot_volume_and_temperature(df_analytics[i])
#     plot_acf_pacf_of_series(df_analytics[i])
#     print("\n---------------------------------\n")

Helper functions to plot permutation importance and learning curve

In [ ]:
# This functions parameters are:
# - model: The trained machine learning model for which permutation importance will be calculated and plotted.
# - X_train: The feature data used for training the model.
# - y_train: The target variable corresponding to the feature data.
# - n_repeats: The number of times to permute a feature.
def plot_permutation_importance(model, X_train, y_train, n_repeats=5):
    result = permutation_importance(
        model,
        X_train,
        y_train,
        n_repeats=n_repeats,
        random_state=42,
        n_jobs=-1
    )

    perm_df = pd.DataFrame({
        "feature": X_train.columns,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std
    }).sort_values("importance_mean", ascending=False)

    plt.figure(figsize=(8, max(4, 0.4 * len(perm_df.head(20)))))
    plt.barh(perm_df.head(20)['feature'], perm_df.head(20)['importance_mean'])
    plt.gca().invert_yaxis()
    plt.xlabel('Permutation Importance (mean drop in score)')
    plt.title('Permutation Feature Importance')
    plt.tight_layout()
    plt.show()

    return perm_df

# function to plot learning curve
# This functions parameters are:
# - model: The machine learning model for which the learning curve will be plotted.
# - X: The feature data used for training the model.
# - y: The target variable corresponding to the feature data.
# - title: The title for the learning curve plot.
def plot_learning_curve(model, X, y, title="Learning Curve"):
    plt.figure(figsize=(10, 6))
    LearningCurveDisplay.from_estimator(
        model,
        X,
        y,
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=2,
        n_jobs=-1,
        scoring='neg_mean_absolute_error'
    )
    plt.grid()
    plt.show()


Training Function

In [ ]:
def train_xgb_up_to_cutoff(df_general, target_day, max_depth = 6, drop_appliances = None):
    prev_day = target_day - timedelta(days=1)
    cutoff_dt = datetime.combine(prev_day, time(hour=10))

    df_train = df_general[df_general.Date <= cutoff_dt].copy()

    lag_cols = [c for c in df_train.columns if c.startswith("lag-")]
    df_train = df_train.dropna(subset=lag_cols + ['Volume'])

    X_train = df_train.drop(columns=['Date', 'Volume'])
    if drop_appliances is not None:
        X_train = X_train.drop(columns=drop_appliances, errors='ignore')
    y_train = df_train['Volume']

    model = xgb.XGBRegressor(
        device='cpu',
        n_estimators=200,
        max_depth=max_depth, 
        learning_rate=0.05,
        subsample=1.0,
        colsample_bytree=0.8
    )
    model.fit(X_train, y_train)

    # plot_learning_curve(model, X_train, y_train)
    # if drop_appliances is None:
    #     plot_permutation_importance(model, X_train, y_train)
    
    return model


Main function to train the model and make predictions

In [ ]:
# Predicts all 24 hours of target_day, using only data up to previous day at 10:00
# This function parameters are:
# - df_general: The general dataframe containing the data.
# - temp_index: The temperature index for the location.
# - target_day: The day for which to make predictions.
# - max_depth: The maximum depth for the model.
# - model: The trained model to use for predictions
# - drop_appliances: List of appliance columns to drop from the features (if any).
# - num: The house number (used to determine which appliances to include in the prediction features).
def predict_day_hourly(df_general, temp_index, target_day, max_depth = 6, model=None, drop_appliances = None, num = 0):
    
    if model is None:
        print("Training new model...")
        model = train_xgb_up_to_cutoff(df_general, target_day, max_depth, drop_appliances=drop_appliances)

    prev_day = target_day - timedelta(days=1)
    cutoff_dt = datetime.combine(prev_day, time(hour=10))

    # Base prediction frame
    df_pred_base = get_weather_for_day(target_day, temp_index, drop_appliances=drop_appliances, num=num)
    df_pred_base = add_date_scalar(df_pred_base)


    df_hist_for_lags = df_general[df_general.Date <= cutoff_dt].copy()
    df_for_lags = pd.concat([df_hist_for_lags, df_pred_base], ignore_index=True)
    df_for_lags = df_for_lags.sort_values('Date').drop_duplicates(subset=['Date'])
    df_for_lags = prepare_data_with_lags(df_for_lags, lag_start=1, lag_end=7, base_offset_days=2, dropna=False)

    mask_target = df_for_lags['Date'].dt.date == target_day
    df_pred_features = df_for_lags.loc[mask_target].copy()

    lag_cols = [c for c in df_pred_features.columns if c.startswith("lag-")]
    df_pred_features = df_pred_features.dropna(subset=lag_cols)

    # Prediction
    X_pred = df_pred_features.drop(columns=['Date', 'Volume'], errors='ignore')
    if drop_appliances is not None:
        X_pred = X_pred.drop(columns=drop_appliances, errors='ignore')
    df_pred_features['Predicted'] = model.predict(X_pred)
    df_result = df_pred_features[['Date', 'Predicted']].sort_values('Date').reset_index(drop=True)
    
    return df_result


Main Functions for Ablation Study

In [ ]:
# This function runs the appliance ablation study by training models with each appliance dropped and evaluating their performance on a daily and hourly basis.
# This function parameters are:
# - df_general: The general dataframe containing the data.
# - temp_index: The temperature index for the location.
# - start_day: The starting day for the evaluation.
# - n_days: The number of days to evaluate.
# - max_depth: The maximum depth for the XGBoost model.
# - appliances: The list of appliance columns to consider for ablation.
def run_appliance_ablation(
    df_general,
    temp_index,
    start_day,
    n_days=15,
    max_depth=6,
    appliances=APPLIANCE_COLS
):
    results_day = []
    results_hour = []

    scenarios = [("baseline", None)] + [(f"drop_{a}", [a]) for a in appliances]

    for scenario_name, drop_list in scenarios:
        print(f"\n=== Scenario: {scenario_name} ===")

        model = train_xgb_up_to_cutoff(
            df_general,
            start_day,
            max_depth=max_depth,
            drop_appliances=drop_list
        )
        for d in range(n_days):
            target_day = start_day + timedelta(days=d)

            df_forecast = predict_day_hourly(
                df_general,
                temp_index,
                target_day,
                max_depth=max_depth,
                model=model,
                drop_appliances=drop_list
            )

            df_actual = df_general[
                df_general['Date'].dt.date == target_day
            ][['Date', 'Volume']].reset_index(drop=True)

            df_eval = pd.merge(df_forecast, df_actual, on='Date', how='inner')
            df_eval['Residual'] = df_eval['Volume'] - df_eval['Predicted']

            df_eval = add_date_scalar(df_eval)
            df_eval = add_type_day(df_eval)

            rmse = float(np.sqrt(np.mean(df_eval['Residual'] ** 2)))
            mae = float(np.mean(np.abs(df_eval['Residual'])))

            results_day.append({
                "Scenario": scenario_name,
                "Dropped": None if drop_list is None else drop_list[0],
                "Day": target_day,
                "RMSE": rmse,
                "MAE": mae
            })

            hourly = (
                df_eval
                .groupby(["Hour", "Weekday", "TypeDay"], as_index=False)
                .agg(
                    RMSE=("Residual", lambda x: float(np.sqrt(np.mean(x**2)))),
                    MAE=("Residual", lambda x: float(np.mean(np.abs(x))))
                )
            )
            hourly["Scenario"] = scenario_name
            hourly["Dropped"] = None if drop_list is None else drop_list[0]
            hourly["Day"] = target_day
            results_hour.append(hourly)

    df_day_metrics = pd.DataFrame(results_day)
    df_hour_metrics = pd.concat(results_hour, ignore_index=True)
    return df_day_metrics, df_hour_metrics


# This function runs all combinations of included appliances (from 0 to all) and evaluates their performance on a daily and hourly basis. It allows for limiting the number of scenarios and optionally including the empty set (no appliances).
# This function parameters are:
# - df_general: The general dataframe containing the data.
# - temp_index: The temperature index for the location.
# - start_day: The starting day for the evaluation.
# - n_days: The number of days to evaluate.
# - max_depth: The maximum depth for the XGBoost model.
# - appliances: The list of appliance columns to consider for combinations.
# - include_empty: Whether to include the scenario with no appliances included.
# - limit_scenarios: An optional limit on the number of scenarios to evaluate
def run_all_appliance_combinations(
    df_general,
    temp_index,
    start_day,
    n_days=15,
    max_depth=6,
    appliances=APPLIANCE_COLS,
    include_empty=False,          
    limit_scenarios=None,
    num=0
):

    results_day = []
    results_hour = []

    subset_sizes = range(0 if include_empty else 1, len(appliances) + 1)
    all_subsets = []
    for r in subset_sizes:
        all_subsets.extend(itertools.combinations(appliances, r))

    if limit_scenarios is not None:
        all_subsets = all_subsets[:limit_scenarios]

    for included in all_subsets:
        included = list(included)
        dropped = [a for a in appliances if a not in included]

        scenario_name = "incl_" + ("none" if len(included) == 0 else "_".join([a.replace("Appliance", "A") for a in included]))
        print(f"\n=== Scenario: {scenario_name} (included={len(included)}, dropped={len(dropped)}) ===")

        model = train_xgb_up_to_cutoff(
            df_general,
            start_day,
            max_depth=max_depth,
            drop_appliances=dropped if len(dropped) > 0 else None
        )

        for d in range(n_days):
            target_day = start_day + timedelta(days=d)

            df_forecast = predict_day_hourly(
                df_general,
                temp_index,
                target_day,
                max_depth=max_depth,
                model=model,
                drop_appliances=dropped if len(dropped) > 0 else None,
                num=num
            )

            df_actual = df_general[
                df_general['Date'].dt.date == target_day
            ][['Date', 'Volume']].reset_index(drop=True)

            df_eval = pd.merge(df_forecast, df_actual, on='Date', how='inner')
            df_eval['Residual'] = df_eval['Volume'] - df_eval['Predicted']

            df_eval = add_date_scalar(df_eval)
            df_eval = add_type_day(df_eval)

            rmse = float(np.sqrt(np.mean(df_eval['Residual'] ** 2)))
            mae = float(np.mean(np.abs(df_eval['Residual'])))

            results_day.append({
                "Scenario": scenario_name,
                "Included": ",".join(included) if len(included) else "",
                "Dropped": ",".join(dropped) if len(dropped) else "",
                "n_included": len(included),
                "Day": target_day,
                "RMSE": rmse,
                "MAE": mae
            })

            hourly = (
                df_eval
                .groupby(["Hour", "Weekday", "TypeDay"], as_index=False)
                .agg(
                    RMSE=("Residual", lambda x: float(np.sqrt(np.mean(x**2)))),
                    MAE=("Residual", lambda x: float(np.mean(np.abs(x))))
                )
            )
            hourly["Scenario"] = scenario_name
            hourly["Included"] = ",".join(included) if len(included) else ""
            hourly["Dropped"] = ",".join(dropped) if len(dropped) else ""
            hourly["n_included"] = len(included)
            hourly["Day"] = target_day
            results_hour.append(hourly)

    df_day_metrics = pd.DataFrame(results_day)
    df_hour_metrics = pd.concat(results_hour, ignore_index=True)
    return df_day_metrics, df_hour_metrics

# This function calculates the marginal contribution of each appliance to the performance metric (e.g., MAE) along with the standard deviation of that contribution across different subsets of included appliances.
# This function parameters are:
# - df_day_metrics: The DataFrame containing the daily performance metrics for each scenario.
# - appliances: The list of appliance columns to consider for calculating marginal contributions.
# - metric: The performance metric to use for calculating marginal contributions (default is "MAE").
def marginal_contribution_with_std(df_day_metrics, appliances=None, metric="MAE"):
    df = df_day_metrics.copy()
    df["IncludedSet"] = df["Included"].fillna("").apply(lambda s: frozenset([x for x in s.split(",") if x]))
    perf = df.groupby("IncludedSet")[metric].mean().to_dict()

    rows = []
    for a in appliances:
        deltas = [perf[S | {a}] - perf[S] for S in perf if (a not in S and (S | {a}) in perf)]
        rows.append({
            "Appliance": a,
            "marginal_mean": float(np.mean(deltas)) if deltas else np.nan,
            "marginal_std": float(np.std(deltas, ddof=1)) if len(deltas) > 1 else np.nan,
            "n": int(len(deltas))
        })
    return pd.DataFrame(rows).sort_values("marginal_mean")


Main Loop for Ablation Study

In [ ]:
metrics_summary = []
meta_metrics = []
best_models = []
big_eval = pd.DataFrame()

train_date = date(2015, 4, 15)

for selected in range(len(df_generals)):

    print(f"Results for House_{house_num[selected]}:")
    df_general = df_generals[selected]
    temp_index = temp_indexs[selected]
    df_day_metrics_all, df_hour_metrics_all = run_all_appliance_combinations(
        df_general=df_general,
        temp_index=temp_index,
        start_day=train_date,
        n_days=15,
        max_depth=6,
        include_empty=True,
        limit_scenarios=None,
        num=house_num[selected]
    )

    best = (
        df_day_metrics_all
        .groupby(["Scenario","Included","n_included"], as_index=False)["MAE"]
        .mean()
        .sort_values("MAE")
    )

    print(best.head(20))
    marginal_importance = marginal_contribution_with_std(df_day_metrics_all, appliances=APPLIANCE_COLS, metric="MAE")
    print("\nMarginal contribution of each appliance (from combos):")
    print(marginal_importance)

    

Results for House_12:

=== Scenario: incl_none (included=0, dropped=9) ===

=== Scenario: incl_A1 (included=1, dropped=8) ===

=== Scenario: incl_A2 (included=1, dropped=8) ===

=== Scenario: incl_A3 (included=1, dropped=8) ===

=== Scenario: incl_A4 (included=1, dropped=8) ===

=== Scenario: incl_A5 (included=1, dropped=8) ===

=== Scenario: incl_A6 (included=1, dropped=8) ===

=== Scenario: incl_A7 (included=1, dropped=8) ===

=== Scenario: incl_A8 (included=1, dropped=8) ===

=== Scenario: incl_A9 (included=1, dropped=8) ===

=== Scenario: incl_A1_A2 (included=2, dropped=7) ===

=== Scenario: incl_A1_A3 (included=2, dropped=7) ===

=== Scenario: incl_A1_A4 (included=2, dropped=7) ===

=== Scenario: incl_A1_A5 (included=2, dropped=7) ===

=== Scenario: incl_A1_A6 (included=2, dropped=7) ===

=== Scenario: incl_A1_A7 (included=2, dropped=7) ===

=== Scenario: incl_A1_A8 (included=2, dropped=7) ===

=== Scenario: incl_A1_A9 (included=2, dropped=7) ===

=== Scenario: incl_A2_A3 (include

KeyboardInterrupt: 

Save to File

In [ ]:
df_day_metrics_all.to_csv(f"Predictions/ablation_results_house_{house_num[selected]}.csv", index=False) 
df_hour_metrics_all.to_csv(f"Predictions/ablation_results_hourly_house_{house_num[selected]}.csv", index=False)
